In [0]:
%sql
create volume databricks_prajakta.bronze.autovol

# **Autoloader Query** 

In [0]:
df = spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "csv") \
  .option("cloudFiles.schemaLocation","/Volumes/databricks_prajakta/bronze/autovol/destination/checkpoint/")\
  .option("cloudFiles.schemaEvolutionMode", "rescue") \
  .load("/Volumes/databricks_prajakta/bronze/autovol/raw/")

In [0]:
df.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation","/Volumes/databricks_prajakta/bronze/autovol/destination/checkpoint/")\
    .trigger(once=True)\
    .start("/Volumes/databricks_prajakta/bronze/autovol/destination/data/")

In [0]:
df = spark.read.format("delta")\
               .load("/Volumes/databricks_prajakta/bronze/autovol/destination/data/")

display(df)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *



In [0]:
rescued_schema = StructType()\
    .add("discount", StringType())\
    .add("payment_method", StringType())

#Parse the stringfied  JSON Column
df = df.withColumn("rescued_struct",from_json(col("_rescued_data"),rescued_schema))

#Extract individual field
df = df.withColumn("rescued_discount",col("rescued_struct.discount"))\
       .withColumn("rescued_payment_method",col("rescued_struct.payment_method"))
    

In [0]:
display(df)